In [1]:
"""
============================================================
CIFAR-10 with Label Noise: LW-AdaDelta vs Baselines
40% Noise Rate Only — Continuation of 20% Results
============================================================
"""

import os
import pickle
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, Subset
import torchvision
import torchvision.transforms as transforms
import numpy as np
import matplotlib.pyplot as plt
from collections import deque
import math
import time
from copy import deepcopy
from scipy import stats

# ============================================================
# LW-AdaDelta Optimizer
# ============================================================

class LWAdaDelta(torch.optim.Optimizer):
    """Local-Window AdaDelta with temporal smoothing"""

    def __init__(self, params, rho=0.9, eps=1e-6, k=2, tau=1.0):
        defaults = dict(rho=rho, eps=eps, k=k, tau=tau)
        super().__init__(params, defaults)
        self._weight_cache = {}

    def _weights(self, k, tau, device):
        key = (k, tau, device)
        if key not in self._weight_cache:
            w = torch.tensor(
                [math.exp(-i / tau) for i in range(k)],
                device=device
            )
            self._weight_cache[key] = w / w.sum()
        return self._weight_cache[key]

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            rho, eps, k, tau = group["rho"], group["eps"], group["k"], group["tau"]
            for p in group["params"]:
                if p.grad is None:
                    continue
                grad  = p.grad
                state = self.state[p]
                if len(state) == 0:
                    state["Eg2"]  = torch.zeros_like(p)
                    state["Edx2"] = torch.zeros_like(p)
                    state["buf"]  = deque(maxlen=k)
                Eg2, Edx2, buf = state["Eg2"], state["Edx2"], state["buf"]
                Eg2.mul_(rho).addcmul_(grad, grad, value=1 - rho)
                rms_dx    = torch.sqrt(Edx2 + eps)
                rms_g     = torch.sqrt(Eg2  + eps)
                delta_raw = -(rms_dx / rms_g) * grad
                buf.append(delta_raw.detach())
                weights = self._weights(len(buf), tau, p.device)
                delta   = torch.zeros_like(p)
                for w, u in zip(weights, reversed(buf)):
                    delta.add_(u, alpha=w.item())
                p.add_(delta)
                Edx2.mul_(rho).addcmul_(delta, delta, value=1 - rho)


class Lion(torch.optim.Optimizer):
    """Lion optimizer"""
    def __init__(self, params, lr=1e-4, betas=(0.9, 0.99), weight_decay=0.0):
        defaults = dict(lr=lr, betas=betas, weight_decay=weight_decay)
        super().__init__(params, defaults)

    @torch.no_grad()
    def step(self):
        for group in self.param_groups:
            for p in group['params']:
                if p.grad is None:
                    continue
                grad    = p.grad
                state   = self.state[p]
                if len(state) == 0:
                    state['exp_avg'] = torch.zeros_like(p)
                exp_avg      = state['exp_avg']
                beta1, beta2 = group['betas']
                if group['weight_decay'] != 0:
                    p.mul_(1 - group['lr'] * group['weight_decay'])
                update = exp_avg.mul(beta1).add(grad, alpha=1 - beta1)
                p.add_(torch.sign(update), alpha=-group['lr'])
                exp_avg.mul_(beta2).add_(grad, alpha=1 - beta2)


# ============================================================
# Noisy CIFAR-10 Dataset
# ============================================================

class NoisyCIFAR10:
    """CIFAR-10 with symmetric label noise"""

    def __init__(self, noise_rate=0.4, seed=42):
        self.noise_rate = noise_rate
        self.seed       = seed

        transform_train = transforms.Compose([
            transforms.RandomCrop(32, padding=4),
            transforms.RandomHorizontalFlip(),
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465),
                                 (0.2470, 0.2435, 0.2616))
        ])
        transform_test = transforms.Compose([
            transforms.ToTensor(),
            transforms.Normalize((0.4914, 0.4822, 0.4465),
                                 (0.2470, 0.2435, 0.2616))
        ])

        self.train_dataset = torchvision.datasets.CIFAR10(
            root='./data', train=True,  download=True,
            transform=transform_train
        )
        self.test_dataset = torchvision.datasets.CIFAR10(
            root='./data', train=False, download=True,
            transform=transform_test
        )

        self._add_label_noise()

        np.random.seed(seed)
        indices = np.random.permutation(len(self.train_dataset))
        self.train_indices = indices[:45000]
        self.val_indices   = indices[45000:]

    def _add_label_noise(self):
        np.random.seed(self.seed)
        n_samples = len(self.train_dataset)
        n_noisy   = int(self.noise_rate * n_samples)

        labels = np.array(self.train_dataset.targets)
        noisy_indices = np.random.choice(n_samples, n_noisy, replace=False)
        for idx in noisy_indices:
            labels[idx] = np.random.randint(0, 10)
        self.train_dataset.targets = labels.tolist()

    def get_loaders(self, batch_size=128):
        train_subset = Subset(self.train_dataset, self.train_indices)
        val_subset   = Subset(self.train_dataset, self.val_indices)

        train_loader = DataLoader(train_subset, batch_size=batch_size,
                                  shuffle=True,  num_workers=2, pin_memory=True)
        val_loader   = DataLoader(val_subset,   batch_size=batch_size,
                                  shuffle=False, num_workers=2, pin_memory=True)
        test_loader  = DataLoader(self.test_dataset, batch_size=batch_size,
                                  shuffle=False, num_workers=2, pin_memory=True)
        return train_loader, val_loader, test_loader


# ============================================================
# ResNet-18
# ============================================================

class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels, out_channels, 3,
                               stride=stride, padding=1, bias=False)
        self.bn1   = nn.BatchNorm2d(out_channels)
        self.conv2 = nn.Conv2d(out_channels, out_channels, 3,
                               padding=1, bias=False)
        self.bn2   = nn.BatchNorm2d(out_channels)
        self.shortcut = nn.Sequential()
        if stride != 1 or in_channels != out_channels:
            self.shortcut = nn.Sequential(
                nn.Conv2d(in_channels, out_channels, 1,
                          stride=stride, bias=False),
                nn.BatchNorm2d(out_channels)
            )

    def forward(self, x):
        out  = F.relu(self.bn1(self.conv1(x)))
        out  = self.bn2(self.conv2(out))
        out += self.shortcut(x)
        return F.relu(out)


class ResNet18(nn.Module):
    def __init__(self, num_classes=10):
        super().__init__()
        self.conv1   = nn.Conv2d(3, 64, 3, padding=1, bias=False)
        self.bn1     = nn.BatchNorm2d(64)
        self.layer1  = self._make_layer(64,  64,  2, stride=1)
        self.layer2  = self._make_layer(64,  128, 2, stride=2)
        self.layer3  = self._make_layer(128, 256, 2, stride=2)
        self.layer4  = self._make_layer(256, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))
        self.fc      = nn.Linear(512, num_classes)

    def _make_layer(self, in_ch, out_ch, n_blocks, stride):
        layers = [BasicBlock(in_ch, out_ch, stride)]
        for _ in range(1, n_blocks):
            layers.append(BasicBlock(out_ch, out_ch, 1))
        return nn.Sequential(*layers)

    def forward(self, x):
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.layer1(out)
        out = self.layer2(out)
        out = self.layer3(out)
        out = self.layer4(out)
        out = self.avgpool(out)
        out = out.view(out.size(0), -1)
        return self.fc(out)


# ============================================================
# Training
# ============================================================

def train_epoch(model, optimizer, train_loader, device):
    model.train()
    total_loss, correct, total = 0.0, 0, 0
    for images, labels in train_loader:
        images, labels = images.to(device), labels.to(device)
        optimizer.zero_grad()
        outputs = model(images)
        loss    = F.cross_entropy(outputs, labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return total_loss / len(train_loader), 100. * correct / total


@torch.no_grad()
def evaluate(model, loader, device):
    model.eval()
    correct, total, total_loss = 0, 0, 0.0
    for images, labels in loader:
        images, labels = images.to(device), labels.to(device)
        outputs     = model(images)
        loss        = F.cross_entropy(outputs, labels)
        total_loss += loss.item()
        _, predicted = outputs.max(1)
        total   += labels.size(0)
        correct += predicted.eq(labels).sum().item()
    return total_loss / len(loader), 100. * correct / total


def train_with_early_stopping(model, optimizer, train_loader, val_loader,
                               device, min_epochs=50, max_epochs=300,
                               patience=15):
    best_val_acc           = 0.0
    best_model_state       = None
    epochs_without_improve = 0
    history = {
        'train_loss': [], 'train_acc': [],
        'val_loss':   [], 'val_acc':   [],
        'epoch_times': []
    }

    for epoch in range(max_epochs):
        t0 = time.time()
        train_loss, train_acc = train_epoch(model, optimizer,
                                            train_loader, device)
        val_loss,   val_acc   = evaluate(model, val_loader, device)
        elapsed = time.time() - t0

        history['train_loss'].append(train_loss)
        history['train_acc'].append(train_acc)
        history['val_loss'].append(val_loss)
        history['val_acc'].append(val_acc)
        history['epoch_times'].append(elapsed)

        if (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}: Train Acc={train_acc:.2f}%  "
                  f"Val Acc={val_acc:.2f}%  Time={elapsed:.1f}s")

        if val_acc > best_val_acc:
            best_val_acc         = val_acc
            best_model_state     = deepcopy(model.state_dict())
            epochs_without_improve = 0
        else:
            epochs_without_improve += 1

        if epoch >= min_epochs and epochs_without_improve >= patience:
            print(f"  Early stopping at epoch {epoch+1} "
                  f"(no improvement for {patience} epochs)")
            break

    if best_model_state is not None:
        model.load_state_dict(best_model_state)

    history['best_val_acc']  = best_val_acc
    history['total_epochs']  = epoch + 1
    return history


# ============================================================
# Optimizer Factory
# ============================================================

def get_optimizer(name, model_params):
    if name == 'SGD':
        return torch.optim.SGD(model_params, lr=0.1,
                               momentum=0.9, weight_decay=5e-4)
    elif name == 'Adam':
        return torch.optim.Adam(model_params, lr=0.001, weight_decay=5e-4)
    elif name == 'AdamW':
        return torch.optim.AdamW(model_params, lr=0.001, weight_decay=5e-4)
    elif name == 'Lion':
        return Lion(model_params, lr=0.0001, weight_decay=5e-4)
    elif name == 'AdaDelta':
        return torch.optim.Adadelta(model_params, rho=0.9)
    elif name == 'LW-AdaDelta':
        return LWAdaDelta(model_params, rho=0.9, k=2, tau=1.0)
    else:
        raise ValueError(f"Unknown optimizer: {name}")


# ============================================================
# Single Experiment
# ============================================================

def run_experiment(noise_rate, optimizer_name, seed, device):
    print(f"\n{'='*60}")
    print(f"Noise={noise_rate*100:.0f}%  Optimizer={optimizer_name}  Seed={seed}")
    print(f"{'='*60}")

    torch.manual_seed(seed)
    np.random.seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)

    noisy_cifar = NoisyCIFAR10(noise_rate=noise_rate, seed=seed)
    train_loader, val_loader, test_loader = noisy_cifar.get_loaders(batch_size=128)

    model = ResNet18(num_classes=10).to(device)
    if torch.cuda.device_count() > 1:
        print(f"  Using {torch.cuda.device_count()} GPUs")
        model = nn.DataParallel(model)

    optimizer = get_optimizer(optimizer_name, model.parameters())

    history = train_with_early_stopping(
        model, optimizer, train_loader, val_loader, device,
        min_epochs=50, max_epochs=300, patience=15
    )

    test_loss, test_acc = evaluate(model, test_loader, device)

    print(f"\n  Final Results:")
    print(f"    Best Val Acc : {history['best_val_acc']:.2f}%")
    print(f"    Test Acc     : {test_acc:.2f}%")
    print(f"    Total Epochs : {history['total_epochs']}")
    print(f"    Avg Time/Ep  : {np.mean(history['epoch_times']):.1f}s")

    return {
        'history':    history,
        'test_acc':   test_acc,
        'test_loss':  test_loss,
        'total_time': sum(history['epoch_times'])
    }


# ============================================================
# Benchmark Runner — 40% noise only, with checkpointing
# ============================================================

CHECKPOINT_PATH = '/kaggle/working/cifar40_checkpoint.pkl'


def run_full_benchmark(device='cuda', seeds=[0, 1, 2]):
    """
    Run 40% noise benchmark with checkpoint save after every
    single experiment. If interrupted, re-running this function
    automatically resumes from where it left off.
    """
    optimizers  = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']
    noise_rate  = 0.4

    # ── Load checkpoint if exists ────────────────────────────────
    if os.path.exists(CHECKPOINT_PATH):
        with open(CHECKPOINT_PATH, 'rb') as f:
            results = pickle.load(f)
        print(f"[INFO] Checkpoint loaded from {CHECKPOINT_PATH}")
        # Count completed experiments
        done = sum(len(results[opt]) for opt in optimizers)
        print(f"[INFO] Already completed: {done} / {len(optimizers)*len(seeds)}")
    else:
        results = {opt: [] for opt in optimizers}
        print("[INFO] No checkpoint found — starting fresh")

    total   = len(optimizers) * len(seeds)
    current = sum(len(results[opt]) for opt in optimizers)

    for opt_name in optimizers:
        for seed in seeds:

            # ── Skip if already done ─────────────────────────────
            completed_seeds = len(results[opt_name])
            if completed_seeds > seed:
                print(f"  Skipping {opt_name} seed={seed} — already done")
                continue

            current += 1
            print(f"\nProgress: {current}/{total}")

            result = run_experiment(noise_rate, opt_name, seed, device)
            results[opt_name].append(result)

            # ── Save checkpoint immediately after each run ────────
            with open(CHECKPOINT_PATH, 'wb') as f:
                pickle.dump(results, f)
            print(f"  ✓ Checkpoint saved ({opt_name} seed={seed} done)")

    return results


# ============================================================
# Analysis
# ============================================================

def analyze_results(results):
    optimizers = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']

    print("\n" + "="*80)
    print("CIFAR-10  —  40% LABEL NOISE  —  RESULTS")
    print("="*80)

    print(f"\n{'Optimizer':<15} {'Test Acc':<22} {'Val Acc':<22} "
          f"{'Epochs':<10} {'Time (s)'}")
    print("-"*75)

    metrics = {}
    for opt_name in optimizers:
        runs      = results[opt_name]
        test_accs = [r['test_acc']                  for r in runs]
        val_accs  = [r['history']['best_val_acc']   for r in runs]
        epochs    = [r['history']['total_epochs']   for r in runs]
        times     = [r['total_time']                for r in runs]

        metrics[opt_name] = {
            'test_mean': np.mean(test_accs),
            'test_std':  np.std(test_accs),
            'val_mean':  np.mean(val_accs),
            'val_std':   np.std(val_accs),
            'ep_mean':   np.mean(epochs),
            'time_mean': np.mean(times),
        }

        print(f"{opt_name:<15} "
              f"{metrics[opt_name]['test_mean']:>6.2f}±{metrics[opt_name]['test_std']:.2f}%    "
              f"{metrics[opt_name]['val_mean']:>6.2f}±{metrics[opt_name]['val_std']:.2f}%    "
              f"{metrics[opt_name]['ep_mean']:>5.0f}    "
              f"{metrics[opt_name]['time_mean']:>8.0f}")

    # ── Head-to-head ──────────────────────────────────────────────
    lw_mean  = metrics['LW-AdaDelta']['test_mean']
    ada_mean = metrics['AdaDelta']['test_mean']
    delta    = lw_mean - ada_mean

    print(f"\n{'='*80}")
    print("LW-ADADELTA vs ADADELTA")
    print(f"{'='*80}")
    print(f"  AdaDelta    : {ada_mean:.2f}%")
    print(f"  LW-AdaDelta : {lw_mean:.2f}%")
    print(f"  Improvement : {delta:+.2f}%  "
          + ("✓ LW-AdaDelta WINS!" if delta > 0 else "✗ AdaDelta wins"))

    # Statistical significance
    lw_accs  = [r['test_acc'] for r in results['LW-AdaDelta']]
    ada_accs = [r['test_acc'] for r in results['AdaDelta']]
    if len(lw_accs) >= 2:
        t_stat, p_value = stats.ttest_rel(lw_accs, ada_accs)
        print(f"\n  t-statistic : {t_stat:.3f}")
        print(f"  p-value     : {p_value:.4f}  "
              + ("✓ Significant (p<0.05)" if p_value < 0.05
                 else "✗ Not significant"))

    # ── Rankings ──────────────────────────────────────────────────
    print(f"\n{'='*80}")
    print("FULL RANKING")
    print(f"{'='*80}")
    ranked = sorted(metrics.items(),
                    key=lambda x: x[1]['test_mean'], reverse=True)
    for rank, (opt, m) in enumerate(ranked, 1):
        marker = " ← LW-AdaDelta" if opt == 'LW-AdaDelta' else \
                 " ← AdaDelta"    if opt == 'AdaDelta'    else ""
        print(f"  {rank}. {opt:<15} {m['test_mean']:.2f}±{m['test_std']:.2f}%{marker}")

    return metrics


# ============================================================
# Visualization
# ============================================================

def plot_results(results, metrics):
    optimizers = ['SGD', 'Adam', 'AdamW', 'Lion', 'AdaDelta', 'LW-AdaDelta']
    colors = {
        'SGD':         '#6b7280',
        'Adam':        '#3b82f6',
        'AdamW':       '#f59e0b',
        'Lion':        '#14b8a6',
        'AdaDelta':    '#ef4444',
        'LW-AdaDelta': '#10b981',
    }

    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    fig.suptitle('CIFAR-10 — 40% Label Noise\nLW-AdaDelta vs Baselines',
                 fontsize=14, fontweight='bold')

    # ── Plot 1: Test accuracy bar chart ───────────────────────────
    ax = axes[0]
    means = [metrics[o]['test_mean'] for o in optimizers]
    stds  = [metrics[o]['test_std']  for o in optimizers]
    bars  = ax.bar(optimizers, means, yerr=stds,
                   color=[colors[o] for o in optimizers],
                   alpha=0.85, edgecolor='black',
                   linewidth=1.2, capsize=5)
    for bar, opt in zip(bars, optimizers):
        if opt in ('LW-AdaDelta', 'AdaDelta'):
            bar.set_linewidth(2.5)
    for bar, m, s in zip(bars, means, stds):
        ax.text(bar.get_x() + bar.get_width()/2,
                m + s + 0.2, f'{m:.2f}%',
                ha='center', va='bottom', fontsize=8, fontweight='bold')
    ax.set_ylabel('Test Accuracy (%)', fontsize=11)
    ax.set_title('Test Accuracy @ 40% Noise', fontsize=12, fontweight='bold')
    ax.set_ylim([max(0, min(means) - 5), min(100, max(means) + 5)])
    ax.grid(True, alpha=0.3, axis='y')
    plt.setp(ax.get_xticklabels(), rotation=20, ha='right', fontsize=9)

    # ── Plot 2: Per-seed delta (LW vs AdaDelta) ───────────────────
    ax = axes[1]
    lw_accs  = [r['test_acc'] for r in results['LW-AdaDelta']]
    ada_accs = [r['test_acc'] for r in results['AdaDelta']]
    diffs    = [lw - ada for lw, ada in zip(lw_accs, ada_accs)]
    labels   = [f'Seed {i}' for i in range(len(diffs))]
    bar_c    = ['#10b981' if d > 0 else '#ef4444' for d in diffs]
    b        = ax.bar(labels, diffs, color=bar_c, alpha=0.85,
                      edgecolor='black', linewidth=1.5)
    ax.axhline(0, color='black', linestyle='--', linewidth=1)
    for bar, d in zip(b, diffs):
        va  = 'bottom' if d >= 0 else 'top'
        off = max(abs(d) * 0.05, 0.05)
        off = off if d >= 0 else -off
        ax.text(bar.get_x() + bar.get_width()/2, d + off,
                f'{d:+.2f}%', ha='center', va=va,
                fontsize=11, fontweight='bold')
    mean_d = np.mean(diffs)
    ax.axhline(mean_d, color='purple', linestyle=':', linewidth=2,
               label=f'Mean Δ={mean_d:+.2f}%')
    ax.legend(fontsize=10)
    ax.set_ylabel('Δ Accuracy (LW-AdaDelta − AdaDelta)', fontsize=11)
    ax.set_title('Per-Seed Improvement over AdaDelta',
                 fontsize=12, fontweight='bold')
    ax.grid(True, alpha=0.3, axis='y')

    # ── Plot 3: Val accuracy curves (median seed) ─────────────────
    ax  = axes[2]
    med = len(results['Adam']) // 2
    for opt in optimizers:
        curve = results[opt][med]['history']['val_acc']
        lw = 2.5 if opt in ('LW-AdaDelta', 'AdaDelta') else 1.2
        al = 1.0 if opt in ('LW-AdaDelta', 'AdaDelta') else 0.55
        ax.plot(curve, label=opt, color=colors[opt],
                linewidth=lw, alpha=al)
    ax.set_xlabel('Epoch', fontsize=11)
    ax.set_ylabel('Validation Accuracy (%)', fontsize=11)
    ax.set_title('Val Accuracy Curve (Median Seed)',
                 fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.grid(True, alpha=0.3)

    plt.tight_layout()
    out = '/kaggle/working/cifar10_40pct_noise_results.png'
    plt.savefig(out, dpi=300, bbox_inches='tight')
    print(f"\n✓ Plot saved: {out}")
    plt.close()


# ============================================================
# Main
# ============================================================

if __name__ == '__main__':

    device = 'cuda' if torch.cuda.is_available() else 'cpu'
    print(f"Using device: {device}")
    print(f"GPUs available: {torch.cuda.device_count()}")

    print("\n" + "="*80)
    print("CIFAR-10 — 40% Label Noise Benchmark")
    print("="*80)
    print("Noise rate    : 40%")
    print("Seeds         : 3")
    print("Early stopping: patience=15, min_epochs=50, max_epochs=300")
    print("Checkpoint    : saved after every experiment")
    print("="*80)

    results = run_full_benchmark(device=device, seeds=[0, 1, 2])

    metrics = analyze_results(results)
    plot_results(results, metrics)

    # Final save
    with open('/kaggle/working/cifar40_final_results.pkl', 'wb') as f:
        pickle.dump(results, f)

    print("\n" + "="*80)
    print("BENCHMARK COMPLETE!")
    print("="*80)
    print("\n✓ Results saved to /kaggle/working/cifar40_final_results.pkl")
    print("✓ Plot saved to /kaggle/working/cifar10_40pct_noise_results.png")
    print("\n>>> STOP THE SESSION NOW: Run menu → Stop Session <<<")

Using device: cuda
GPUs available: 2

CIFAR-10 — 40% Label Noise Benchmark
Noise rate    : 40%
Seeds         : 3
Early stopping: patience=15, min_epochs=50, max_epochs=300
Checkpoint    : saved after every experiment
[INFO] No checkpoint found — starting fresh

Progress: 1/18

Noise=40%  Optimizer=SGD  Seed=0


100%|██████████| 170M/170M [00:05<00:00, 33.9MB/s] 


  Using 2 GPUs
  Epoch  10: Train Acc=44.11%  Val Acc=63.88%  Time=26.6s
  Epoch  20: Train Acc=47.23%  Val Acc=66.04%  Time=26.5s
  Epoch  30: Train Acc=48.47%  Val Acc=67.62%  Time=26.9s
  Epoch  40: Train Acc=49.19%  Val Acc=77.22%  Time=26.4s
  Epoch  50: Train Acc=49.45%  Val Acc=74.98%  Time=26.4s
  Early stopping at epoch 55 (no improvement for 15 epochs)

  Final Results:
    Best Val Acc : 77.22%
    Test Acc     : 77.42%
    Total Epochs : 55
    Avg Time/Ep  : 26.6s
  ✓ Checkpoint saved (SGD seed=0 done)

Progress: 2/18

Noise=40%  Optimizer=SGD  Seed=1
  Using 2 GPUs
  Epoch  10: Train Acc=44.30%  Val Acc=67.92%  Time=26.7s
  Epoch  20: Train Acc=47.75%  Val Acc=72.18%  Time=26.4s
  Epoch  30: Train Acc=48.87%  Val Acc=64.94%  Time=26.6s
  Epoch  40: Train Acc=49.62%  Val Acc=74.42%  Time=26.7s
  Epoch  50: Train Acc=49.79%  Val Acc=76.04%  Time=26.5s
  Epoch  60: Train Acc=50.14%  Val Acc=75.18%  Time=26.3s
  Early stopping at epoch 62 (no improvement for 15 epochs)

  Fin